# classification_dataset

### Imports

In [1]:
import os
import sys
import json

import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon

import keras
import cv2 as cv

2025-07-06 12:40:43.904847: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-06 12:40:43.911383: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751816443.918752  358132 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751816443.921070  358132 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1751816443.926857  358132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPU Available:", gpus)
print("cuDNN Enabled:", tf.test.is_built_with_cuda())

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
cuDNN Enabled: True


### Definitions

In [6]:
IMAGES_PATH = '../../media/data/input_color/'

sys.path.insert(0, "../../")
from config import MEDIA_PATH, CROPPED_PATH, MODELS_PATH

#Configuration
BATCH_SIZE = 32

# Paths
CROPS_PATHS_TRAIN = os.path.join(CROPPED_PATH, 'classification', 'train')
CROPS_PATHS_VALID = os.path.join(CROPPED_PATH, 'classification', 'validation')
MODEL_PATH = os.path.join(MODELS_PATH, 'supervised', 'classifier.keras')
OUTPUT_PATH = os.path.join(MEDIA_PATH, 'cropped_images', 'dividing')


### Functions

In [15]:
def process_images_in_batches(string_list, batch_size=10):
  """
  Processes a list of strings in batches of a specified size.

  Args:
    string_list: The list of strings to process.
    batch_size: The size of each batch.

  Yields:
    A batch of strings.
  """
  for i in range(0, len(string_list), batch_size):
    batch_num = i // batch_size + 1  # Calculate batch number (1-indexed)
    yield batch_num, string_list[i:i + batch_size]

def predict_cell(model, images_batch, color_type):
  """
  Given an image batch it returns the predictions of the batch with the given model.

  Args:
    model: keras model to use.
    image_path: path to the folder where the images are.
    images_batch: list of the image names to include in the batch

  Returns:
    A list of predictions.
  """

  images = []
  for image in images_batch:
      img = cv.imread(image, color_type)
      img = cv.resize(img, (128, 128))
      img = img / 255.0
      images.append(img)
  
  batch = np.stack(images)
  if color_type == cv.IMREAD_GRAYSCALE:\
    # Add missing channel
    batch = np.expand_dims(batch, axis=-1).astype(np.float32)

  prediction = model.predict(batch, verbose=0)
  prediction = tf.nn.softmax(prediction, axis=-1)
  return prediction

def list_files(directory):
    all_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            all_files.append(os.path.join(root, file))
    return all_files

### List of elements to use

In [16]:
train = sorted(list_files(CROPS_PATHS_TRAIN)) #Paths to the csv of SAM detections of each image
valid = sorted(list_files(CROPS_PATHS_VALID)) #Paths to the csv of SAM detections of each image
all_crops = train + valid

model = keras.models.load_model(MODEL_PATH)
print(model.input[0].shape)
if model.input[0].shape[-1] == 1:
    color_type = cv.IMREAD_GRAYSCALE
else:
    color_type = cv.IMREAD_COLOR

(None, 128, 128, 1)


### Generate dataset

In [ ]:
dividing_cells = []
for idx, batch in process_images_in_batches(all_crops, batch_size=BATCH_SIZE): #Read the images in batch_size batches
    print(f"Processing batch {idx/len(all_crops)/BATCH_SIZE}", end='\r')
    batch_prediction = predict_cell(model, images_batch=batch, color_type=color_type)

    is_dividing = 1-np.argmax(batch_prediction, axis=1).astype(bool)

    for idx, crop in enumerate(batch):
        if is_dividing[idx]:
            dividing_cells.append(crop)

I0000 00:00:1751816742.489429  358510 service.cc:152] XLA service 0x7e5ab8016bc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1751816742.489443  358510 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2025-07-06 12:45:42.493891: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1751816742.523719  358510 cuda_dnn.cc:529] Loaded cuDNN version 90701
I0000 00:00:1751816743.042041  358510 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [18]:
os.makedirs(OUTPUT_PATH, exist_ok=True)
for img_path in dividing_cells:
    shutil.copy(img_path, OUTPUT_PATH)